In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251118_170645"


In [2]:
# 1. Setup de caminhos locais
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
RAW_PATH: C:\Users\ander\OneDrive\TCC_USP\data_raw


In [3]:
# 2. Parametros globais e utilitarios
import pandas as pd
import yfinance as yf
from datetime import datetime

PERIODO = cfg.get_periodo_estudo()
START_DATE = PERIODO["start"]
END_DATE = PERIODO["end"]
TICKERS = {"ibovespa": "^BVSP", "bova11": "BOVA11.SA"}

os.makedirs(RAW_PATH, exist_ok=True)
print(f"Periodo configurado: {START_DATE} -> {END_DATE}")
print("Tickers alvo:", TICKERS)


Periodo configurado: 2018-01-01 -> 2025-01-31
Tickers alvo: {'ibovespa': '^BVSP', 'bova11': 'BOVA11.SA'}


In [4]:
# 3. Funcoes para download via yfinance
from typing import Tuple

def download_ticker(name: str, ticker: str, start: str, end: str) -> Tuple[Path, pd.DataFrame]:
    print(f"Baixando {ticker} ({start} -> {end})...")
    data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if data.empty:
        raise ValueError(f"Sem dados retornados para {ticker} no periodo especificado.")
    data = data.reset_index().rename(columns={"Date": "day", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Adj Close": "adj_close", "Volume": "volume"})
    data["source_ticker"] = ticker
    target_path = Path(RAW_PATH) / f"{name}.csv"
    data.to_csv(target_path, index=False)
    print(f"Salvo em {target_path}")
    return target_path, data


In [5]:
# 4. Executar downloads para todos os tickers
summary_rows = []
for name, ticker in TICKERS.items():
    path, df = download_ticker(name, ticker, START_DATE, END_DATE)
    summary_rows.append({"arquivo": path.name, "ticker": ticker, "records": len(df), "inicio": df["day"].min(), "fim": df["day"].max()})
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


Baixando ^BVSP (2018-01-01 -> 2025-01-31)...


Salvo em C:\Users\ander\OneDrive\TCC_USP\data_raw\ibovespa.csv
Baixando BOVA11.SA (2018-01-01 -> 2025-01-31)...


Salvo em C:\Users\ander\OneDrive\TCC_USP\data_raw\bova11.csv


,arquivo,ticker,records,inicio,fim
0,ibovespa.csv,^BVSP,1758,2018-01-02,2025-01-30
1,bova11.csv,BOVA11.SA,1741,2018-01-02,2025-01-30


In [6]:
# 5. Validar cobertura temporal dos dados baixados
import pandas as pd

print("\n" + "="*60)
print("VALIDAÇÃO DE COBERTURA TEMPORAL (2018-2025)")
print("="*60)

expected_start = pd.to_datetime(START_DATE)
expected_end = pd.to_datetime(END_DATE)
validation_errors = []

for name, ticker in TICKERS.items():
    filepath = Path(RAW_PATH) / f"{name}.csv"
    if not filepath.exists():
        validation_errors.append(f"❌ {name}: Arquivo não encontrado")
        continue
    
    df = pd.read_csv(filepath)
    df["day"] = pd.to_datetime(df["day"])
    
    actual_start = df["day"].min()
    actual_end = df["day"].max()
    n_records = len(df)
    
    print(f"\n{name} ({ticker}):")
    print(f"  Esperado: {expected_start.date()} → {expected_end.date()}")
    print(f"  Obtido:   {actual_start.date()} → {actual_end.date()}")
    print(f"  Registros: {n_records}")
    
    # Validações críticas
    if actual_start > expected_start:
        msg = f"⚠️ {name}: Início dos dados ({actual_start.date()}) POSTERIOR ao esperado ({expected_start.date()})"
        print(f"  {msg}")
        validation_errors.append(msg)
    
    if actual_end < expected_end:
        msg = f"⚠️ {name}: Fim dos dados ({actual_end.date()}) ANTERIOR ao esperado ({expected_end.date()})"
        print(f"  {msg}")
        validation_errors.append(msg)
    
    if n_records < 1000:
        msg = f"⚠️ {name}: Dataset muito pequeno ({n_records} registros) - esperado ~1700 dias úteis"
        print(f"  {msg}")
        validation_errors.append(msg)
    
    if n_records >= 1000:
        print(f"  ✅ Cobertura adequada")

# Resumo final
print("\n" + "="*60)
if validation_errors:
    print("❌ VALIDAÇÃO FALHOU - Problemas encontrados:")
    for err in validation_errors:
        print(f"  {err}")
    print("\n⚠️ AVISO: Os dados podem não cobrir completamente o período 2018-2025")
    print("   Isso pode ocorrer por:")
    print("   - Limitações da API yfinance")
    print("   - Feriados/pregões não ocorridos")
    print("   - Dados ainda não disponíveis para datas futuras")
else:
    print("✅ VALIDAÇÃO PASSOU - Todos os dados cobrem o período esperado")

print("="*60)

# 6. Listar arquivos gerados em data_raw
print("\nArquivos gerados:")
for path in Path(RAW_PATH).glob('*.csv'):
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name}: {size_kb:0.1f} KB")



VALIDAÇÃO DE COBERTURA TEMPORAL (2018-2025)

ibovespa (^BVSP):
  Esperado: 2018-01-01 → 2025-01-31
  Obtido:   2018-01-02 → 2025-01-30
  Registros: 1759
  ⚠️ ibovespa: Início dos dados (2018-01-02) POSTERIOR ao esperado (2018-01-01)
  ⚠️ ibovespa: Fim dos dados (2025-01-30) ANTERIOR ao esperado (2025-01-31)
  ✅ Cobertura adequada

bova11 (BOVA11.SA):
  Esperado: 2018-01-01 → 2025-01-31
  Obtido:   2018-01-02 → 2025-01-30
  Registros: 1742
  ⚠️ bova11: Início dos dados (2018-01-02) POSTERIOR ao esperado (2018-01-01)
  ⚠️ bova11: Fim dos dados (2025-01-30) ANTERIOR ao esperado (2025-01-31)
  ✅ Cobertura adequada

❌ VALIDAÇÃO FALHOU - Problemas encontrados:
  ⚠️ ibovespa: Início dos dados (2018-01-02) POSTERIOR ao esperado (2018-01-01)
  ⚠️ ibovespa: Fim dos dados (2025-01-30) ANTERIOR ao esperado (2025-01-31)
  ⚠️ bova11: Início dos dados (2018-01-02) POSTERIOR ao esperado (2018-01-01)
  ⚠️ bova11: Fim dos dados (2025-01-30) ANTERIOR ao esperado (2025-01-31)

⚠️ AVISO: Os dados podem nã